In [3]:
import tkinter as tk
from PIL import Image, ImageTk
from pathlib import Path
import csv
import threading
import numpy as np
from collections import deque
from rembg import remove

# =========================
# CONFIG
# =========================
SCAN_ROOT      = Path(r"G:\My Drive\Machine Learning Group Project\Sampled Images")
REMAINING_ROOT = Path(r"G:\My Drive\Machine Learning Group Project\Remaining Images")
PROCESSED_ROOT = Path(r"G:\My Drive\Machine Learning Group Project\Processed Images")
OUTPUT_CSV     = Path(r"G:\My Drive\Machine Learning Group Project\crop_coordinates.csv")
INVALID_CSV    = Path(r"G:\My Drive\Machine Learning Group Project\invalid_images.csv")

IMAGE_EXTS   = {".png", ".jpg", ".jpeg", ".webp", ".bmp", ".gif", ".tif", ".tiff"}
MAX_SIZE     = 500
TARGET_SIZE  = 256
REQUIRED     = 50
TOLERANCE    = 20
BRUSH_SIZE   = 10
ZOOM_STEP    = 1.25
MAX_ZOOM     = 8.0
MIN_ZOOM     = 1.0
# =========================

# -------------------------
# Load all images
# -------------------------
all_files = sorted([
    p for p in SCAN_ROOT.rglob("*")
    if p.is_file() and p.suffix.lower() in IMAGE_EXTS
])

processed = set()
for csv_path in [OUTPUT_CSV, INVALID_CSV]:
    if csv_path.exists():
        with open(csv_path, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                processed.add(row["full_path"])

queue = [p for p in all_files if str(p) not in processed]

print(f"Found:     {len(all_files):,}")
print(f"Processed: {len(processed):,}")
print(f"Remaining: {len(queue):,}\n")

if not queue:
    print("All images processed!")
    raise SystemExit(0)

# CSV writers
crop_file    = open(OUTPUT_CSV,  "a", newline="", encoding="utf-8")
invalid_file = open(INVALID_CSV, "a", newline="", encoding="utf-8")
crop_writer    = csv.writer(crop_file)
invalid_writer = csv.writer(invalid_file)

if not OUTPUT_CSV.exists() or OUTPUT_CSV.stat().st_size == 0:
    crop_writer.writerow(["file_name", "folder", "width", "height", "dimensions",
                          "left", "top", "right", "bottom", "crop_size",
                          "background_removed", "status", "full_path"])

if not INVALID_CSV.exists() or INVALID_CSV.stat().st_size == 0:
    invalid_writer.writerow(["file_name", "folder", "width", "height",
                              "dimensions", "full_path", "replacement"])


# -------------------------
# Helpers
# -------------------------
def get_replacement(folder_name: str) -> str:
    used = set()
    if INVALID_CSV.exists():
        with open(INVALID_CSV, "r", encoding="utf-8") as f:
            reader = csv.DictReader(f)
            for row in reader:
                if row.get("replacement"):
                    used.add(row["replacement"])
    queued = {str(p) for p in queue}
    remaining_dir = REMAINING_ROOT / folder_name
    if not remaining_dir.exists():
        return ""
    for p in sorted(remaining_dir.iterdir()):
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS:
            if str(p) not in used and str(p) not in queued:
                return str(p)
    return ""


def get_processed_count(folder_name: str) -> int:
    processed_dir = PROCESSED_ROOT / folder_name
    if not processed_dir.exists():
        return 0
    return sum(1 for p in processed_dir.iterdir()
               if p.is_file() and p.suffix.lower() in IMAGE_EXTS)


def get_final_filename(folder_name: str) -> str:
    dest_dir = PROCESSED_ROOT / folder_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    i = 1
    while True:
        candidate = dest_dir / f"Final {i:02d}.png"
        if not candidate.exists():
            return str(candidate)
        i += 1


def resize_to_256(img: Image.Image) -> Image.Image:
    img     = img.convert("RGBA")
    w, h    = img.size
    scale   = min(TARGET_SIZE / w, TARGET_SIZE / h)
    new_w   = int(w * scale)
    new_h   = int(h * scale)
    resized = img.resize((new_w, new_h), Image.LANCZOS)
    canvas  = Image.new("RGBA", (TARGET_SIZE, TARGET_SIZE), (0, 0, 0, 0))
    canvas.paste(resized, ((TARGET_SIZE - new_w) // 2,
                           (TARGET_SIZE - new_h) // 2), resized)
    return canvas


def ensure_50(folder_name: str):
    count  = get_processed_count(folder_name)
    needed = REQUIRED - count
    if needed <= 0:
        return
    added = 0
    for _ in range(needed):
        replacement = get_replacement(folder_name)
        if not replacement:
            break
        repl_path = Path(replacement)
        if str(repl_path) not in processed:
            queue.insert(current_index, repl_path)
            added += 1
    if added > 0:
        status_label.config(
            text=f"⚠️ {folder_name} has {count}/{REQUIRED} — added {added} replacement(s)",
            fg="orange"
        )
    elif count < REQUIRED:
        status_label.config(
            text=f"⚠️ {folder_name} has {count}/{REQUIRED} — no more replacements!",
            fg="red"
        )


# -------------------------
# Touch-up tools
# -------------------------
def color_distance(c1, c2):
    return sum((int(a) - int(b)) ** 2 for a, b in zip(c1[:3], c2[:3])) ** 0.5


def magic_wand(img: Image.Image, x: int, y: int, tolerance: int) -> Image.Image:
    img    = img.convert("RGBA")
    pixels = np.array(img)
    h, w   = pixels.shape[:2]
    target_color = tuple(pixels[y, x])
    if target_color[3] == 0:
        return img
    visited = np.zeros((h, w), dtype=bool)
    queue_  = deque()
    queue_.append((x, y))
    while queue_:
        cx, cy = queue_.popleft()
        if cx < 0 or cx >= w or cy < 0 or cy >= h:
            continue
        if visited[cy, cx]:
            continue
        visited[cy, cx] = True
        current_color = tuple(pixels[cy, cx])
        if current_color[3] == 0:
            continue
        if color_distance(current_color, target_color) <= tolerance:
            pixels[cy, cx] = (0, 0, 0, 0)
            queue_.append((cx + 1, cy))
            queue_.append((cx - 1, cy))
            queue_.append((cx, cy + 1))
            queue_.append((cx, cy - 1))
    return Image.fromarray(pixels, "RGBA")


def color_range_remove(img: Image.Image, x: int, y: int, tolerance: int) -> Image.Image:
    img    = img.convert("RGBA")
    pixels = np.array(img)
    target_color = tuple(pixels[y, x])
    if target_color[3] == 0:
        return img
    diff = pixels[:, :, :3].astype(int) - np.array(target_color[:3], dtype=int)
    dist = np.sqrt((diff ** 2).sum(axis=2))
    mask = (dist <= tolerance) & (pixels[:, :, 3] > 0)
    pixels[mask] = (0, 0, 0, 0)
    return Image.fromarray(pixels, "RGBA")


def restore_pixels(img: Image.Image, orig: Image.Image,
                   x: int, y: int, radius: int) -> Image.Image:
    img    = img.convert("RGBA")
    orig   = orig.convert("RGBA")
    pixels = np.array(img)
    orig_p = np.array(orig)
    h, w   = pixels.shape[:2]
    for py in range(max(0, y - radius), min(h, y + radius)):
        for px in range(max(0, x - radius), min(w, x + radius)):
            if (px - x) ** 2 + (py - y) ** 2 <= radius ** 2:
                pixels[py, px] = orig_p[py, px]
    return Image.fromarray(pixels, "RGBA")


def erase_pixels(img: Image.Image, x: int, y: int, radius: int) -> Image.Image:
    img    = img.convert("RGBA")
    pixels = np.array(img)
    h, w   = pixels.shape[:2]
    for py in range(max(0, y - radius), min(h, y + radius)):
        for px in range(max(0, x - radius), min(w, x + radius)):
            if (px - x) ** 2 + (py - y) ** 2 <= radius ** 2:
                pixels[py, px] = (0, 0, 0, 0)
    return Image.fromarray(pixels, "RGBA")


def render_preview(img: Image.Image):
    global tk_preview_img, preview_scale

    zoomed_size = int(TARGET_SIZE * zoom_level)

    # Build checkered background at zoomed size
    checker = Image.new("RGBA", (zoomed_size, zoomed_size), (255, 255, 255, 255))
    size = 10
    for y in range(0, zoomed_size, size):
        for x in range(0, zoomed_size, size):
            if (x // size + y // size) % 2 == 0:
                for py in range(y, min(y + size, zoomed_size)):
                    for px in range(x, min(x + size, zoomed_size)):
                        checker.putpixel((px, py), (200, 200, 200, 255))

    # Use NEAREST for zoom so pixels stay crisp
    display_resized = img.resize((zoomed_size, zoomed_size), Image.NEAREST)
    if display_resized.mode == "RGBA":
        checker.paste(display_resized, mask=display_resized.split()[3])
        display_final = checker
    else:
        display_final = display_resized

    tk_preview_img = ImageTk.PhotoImage(display_final)

    # Update canvas scroll region
    preview_canvas.config(scrollregion=(0, 0, zoomed_size, zoomed_size))
    preview_canvas.delete("all")
    preview_canvas.create_image(0, 0, anchor="nw", image=tk_preview_img)

    # preview_scale maps canvas coords back to 256x256 image coords
    preview_scale = 1.0 / zoom_level


def canvas_to_image_coords(cx: int, cy: int) -> tuple[int, int]:
    """Convert canvas click coords to 256x256 image coords accounting for zoom and scroll."""
    scroll_x = preview_canvas.xview()[0]
    scroll_y = preview_canvas.yview()[0]
    zoomed_size = int(TARGET_SIZE * zoom_level)
    actual_x = cx + scroll_x * zoomed_size
    actual_y = cy + scroll_y * zoomed_size
    img_x = int(actual_x / zoom_level)
    img_y = int(actual_y / zoom_level)
    img_x = max(0, min(TARGET_SIZE - 1, img_x))
    img_y = max(0, min(TARGET_SIZE - 1, img_y))
    return img_x, img_y


# -------------------------
# State
# -------------------------
current_index    = 0
crop_coords      = None
rect             = None
start_x = start_y = 0
scale            = 1.0
preview_img      = None
final_img        = None
prev_final_img   = None
original_cropped = None
tk_preview_img   = None
processing       = False
pending_save_info = None
touch_up_mode    = None
brush_mode       = False
erase_mode       = False
preview_scale    = 1.0
zoom_level       = 1.0


def reset_tools():
    global touch_up_mode, brush_mode, erase_mode
    touch_up_mode = None
    brush_mode    = False
    erase_mode    = False
    for btn in [wand_btn, range_btn, brush_btn, erase_btn]:
        btn.config(relief="raised", state="disabled")
    undo_btn.config(state="disabled")
    preview_canvas.config(cursor="arrow")


def load_image(index):
    global scale, tk_img, crop_coords, rect, preview_img, final_img
    global prev_final_img, original_cropped, tk_preview_img
    global pending_save_info, zoom_level

    p    = queue[index]
    img  = Image.open(p)
    w, h = img.size

    scale     = min(MAX_SIZE / w, MAX_SIZE / h, 1.0)
    display_w = int(w * scale)
    display_h = int(h * scale)

    display_img = img.resize((display_w, display_h), Image.LANCZOS)
    tk_img      = ImageTk.PhotoImage(display_img)

    orig_canvas.config(width=display_w, height=display_h)
    orig_canvas.delete("all")
    orig_canvas.create_image(0, 0, anchor="nw", image=tk_img)
    orig_canvas.image_info = {"path": p, "w": w, "h": h, "img": img}

    preview_canvas.delete("all")
    preview_canvas.create_text(
        TARGET_SIZE // 2, TARGET_SIZE // 2,
        text="Preview will appear here",
        fill="gray", font=("Arial", 12)
    )

    crop_coords       = None
    rect              = None
    preview_img       = None
    final_img         = None
    prev_final_img    = None
    original_cropped  = None
    tk_preview_img    = None
    pending_save_info = None
    zoom_level        = 1.0

    reset_tools()

    info_label.config(
        text=f"{p.parent.name} / {p.name}  |  {w}x{h}  |  "
             f"{index + 1} of {len(queue)}  |  "
             f"Processed: {get_processed_count(p.parent.name)}/{REQUIRED}"
    )
    coords_label.config(text="No crop set", fg="blue")
    confirm_btn.config(state="disabled")
    save_btn.config(state="disabled")
    redo_btn.config(state="disabled")
    status_label.config(text="")
    root.title(f"{index + 1}/{len(queue)} — {p.name}")


def advance():
    global current_index

    if current_index < len(queue):
        current_folder = queue[current_index].parent.name
        next_folder    = queue[current_index + 1].parent.name \
                         if current_index + 1 < len(queue) else None
        if next_folder != current_folder:
            ensure_50(current_folder)

    current_index += 1
    if current_index >= len(queue):
        info_label.config(text="✅ All images processed!")
        for btn in [confirm_btn, save_btn, redo_btn, skip_btn,
                    invalid_btn, already_btn, wand_btn, range_btn,
                    brush_btn, erase_btn, undo_btn]:
            btn.config(state="disabled")
        orig_canvas.delete("all")
        preview_canvas.delete("all")
        crop_file.close()
        invalid_file.close()
        return
    load_image(current_index)


# -------------------------
# Original canvas interactions
# -------------------------
def on_press(event):
    global start_x, start_y, rect
    start_x, start_y = event.x, event.y
    if rect:
        orig_canvas.delete(rect)


def on_drag(event):
    global rect
    if rect:
        orig_canvas.delete(rect)
    rect = orig_canvas.create_rectangle(
        start_x, start_y, event.x, event.y,
        outline="red", width=2
    )


def on_release(event):
    global crop_coords
    x1 = min(start_x, event.x)
    y1 = min(start_y, event.y)
    x2 = max(start_x, event.x)
    y2 = max(start_y, event.y)

    orig_x1 = int(x1 / scale)
    orig_y1 = int(y1 / scale)
    orig_x2 = int(x2 / scale)
    orig_y2 = int(y2 / scale)

    crop_coords = (orig_x1, orig_y1, orig_x2, orig_y2)
    coords_label.config(
        text=f"Crop: ({orig_x1}, {orig_y1}, {orig_x2}, {orig_y2})  →  "
             f"{orig_x2 - orig_x1}x{orig_y2 - orig_y1}",
        fg="blue"
    )
    confirm_btn.config(state="normal")
    save_btn.config(state="disabled")
    redo_btn.config(state="disabled")


# -------------------------
# Preview canvas interactions
# -------------------------
def on_preview_click(event):
    global final_img, prev_final_img, touch_up_mode

    if final_img is None or touch_up_mode is None:
        return

    px, py = canvas_to_image_coords(event.x, event.y)
    tol    = int(tolerance_var.get())

    prev_final_img = final_img.copy()
    undo_btn.config(state="normal")

    if touch_up_mode == "wand":
        final_img = magic_wand(final_img, px, py, tol)
    elif touch_up_mode == "range":
        final_img = color_range_remove(final_img, px, py, tol)

    render_preview(final_img)
    status_label.config(
        text=f"Touch-up applied ({touch_up_mode}). Save or keep editing.",
        fg="green"
    )


def on_preview_drag(event):
    global final_img, prev_final_img

    if final_img is None:
        return
    if not brush_mode and not erase_mode:
        return

    px, py  = canvas_to_image_coords(event.x, event.y)
    radius  = int(brush_size_var.get())

    if prev_final_img is None:
        prev_final_img = final_img.copy()
        undo_btn.config(state="normal")

    if brush_mode and original_cropped is not None:
        final_img = restore_pixels(final_img, original_cropped, px, py, radius)
    elif erase_mode:
        final_img = erase_pixels(final_img, px, py, radius)

    render_preview(final_img)


def on_preview_scroll(event):
    global zoom_level

    if final_img is None:
        return

    if event.delta > 0 or event.num == 4:
        zoom_level = min(MAX_ZOOM, zoom_level * ZOOM_STEP)
    else:
        zoom_level = max(MIN_ZOOM, zoom_level / ZOOM_STEP)

    render_preview(final_img)
    zoom_label.config(text=f"Zoom: {zoom_level:.1f}x")


# -------------------------
# Tool toggles
# -------------------------
def deactivate_all_tools():
    global touch_up_mode, brush_mode, erase_mode
    touch_up_mode = None
    brush_mode    = False
    erase_mode    = False
    for btn in [wand_btn, range_btn, brush_btn, erase_btn]:
        btn.config(relief="raised")
    preview_canvas.config(cursor="arrow")


def on_wand_toggle():
    global touch_up_mode
    if touch_up_mode == "wand":
        deactivate_all_tools()
        status_label.config(text="")
    else:
        deactivate_all_tools()
        touch_up_mode = "wand"
        wand_btn.config(relief="sunken")
        preview_canvas.config(cursor="crosshair")
        status_label.config(
            text="🪄 Magic Wand — click background in preview", fg="purple"
        )


def on_range_toggle():
    global touch_up_mode
    if touch_up_mode == "range":
        deactivate_all_tools()
        status_label.config(text="")
    else:
        deactivate_all_tools()
        touch_up_mode = "range"
        range_btn.config(relief="sunken")
        preview_canvas.config(cursor="crosshair")
        status_label.config(
            text="🎨 Color Range — click color to remove everywhere", fg="purple"
        )


def on_brush_toggle():
    global brush_mode
    if brush_mode:
        deactivate_all_tools()
        status_label.config(text="")
    else:
        deactivate_all_tools()
        brush_mode = True
        brush_btn.config(relief="sunken")
        preview_canvas.config(cursor="circle")
        status_label.config(
            text="🖌️ Restore Brush — drag to restore original pixels", fg="purple"
        )


def on_erase_toggle():
    global erase_mode
    if erase_mode:
        deactivate_all_tools()
        status_label.config(text="")
    else:
        deactivate_all_tools()
        erase_mode = True
        erase_btn.config(relief="sunken")
        preview_canvas.config(cursor="circle")
        status_label.config(
            text="🧹 Erase Brush — drag to erase pixels", fg="purple"
        )


def on_undo():
    global final_img, prev_final_img
    if prev_final_img is None:
        return
    final_img      = prev_final_img
    prev_final_img = None
    undo_btn.config(state="disabled")
    render_preview(final_img)
    status_label.config(text="↩ Undo applied", fg="gray")


# -------------------------
# Show final preview
# -------------------------
def show_final_preview(source_img: Image.Image, save_info: dict):
    global final_img, tk_preview_img, pending_save_info
    global preview_scale, original_cropped, zoom_level

    zoom_level        = 1.0
    original_cropped  = resize_to_256(source_img)
    final_img         = original_cropped.copy()
    pending_save_info = save_info
    preview_scale     = 1.0

    render_preview(final_img)
    zoom_label.config(text="Zoom: 1.0x")

    status_label.config(
        text=f"Final: {TARGET_SIZE}x{TARGET_SIZE} — Save, Redo, or touch up. "
             f"Scroll to zoom.",
        fg="green"
    )
    save_btn.config(state="normal")
    redo_btn.config(state="normal")
    confirm_btn.config(state="normal")
    for btn in [wand_btn, range_btn, brush_btn, erase_btn]:
        btn.config(state="normal")


# -------------------------
# Preview button
# -------------------------
def on_confirm():
    global processing, preview_img
    if crop_coords is None or processing:
        return

    info = orig_canvas.image_info
    img  = info["img"]
    l, t, r, b = crop_coords
    cropped    = img.crop((l, t, r, b))

    save_info = {
        "path":            info["path"],
        "w":               info["w"],
        "h":               info["h"],
        "crop":            crop_coords,
        "bg_removed":      bg_remove_var.get(),
        "already_correct": False
    }

    if bg_remove_var.get():
        processing = True
        confirm_btn.config(state="disabled")
        status_label.config(text="⏳ Removing background...", fg="orange")
        root.update()

        def run_rembg():
            global processing, preview_img
            try:
                result      = remove(cropped)
                preview_img = result
            except Exception as e:
                preview_img = cropped
                root.after(0, lambda: status_label.config(
                    text=f"⚠️ BG removal failed: {e}", fg="red"
                ))
            finally:
                processing = False
                root.after(0, lambda: show_final_preview(preview_img, save_info))

        threading.Thread(target=run_rembg, daemon=True).start()
    else:
        preview_img = cropped
        show_final_preview(preview_img, save_info)


# -------------------------
# Save
# -------------------------
def on_save():
    global final_img, pending_save_info
    if final_img is None or pending_save_info is None:
        return

    info       = pending_save_info
    p          = info["path"]
    w, h       = info["w"], info["h"]
    l, t, r, b = info["crop"]
    bg_removed = info["bg_removed"]
    final_path = get_final_filename(p.parent.name)
    status     = "cropped_bg_removed" if bg_removed else "cropped"

    final_img.convert("RGBA").save(final_path, "PNG")

    crop_writer.writerow([
        p.name, p.parent.name, w, h, f"{w}x{h}",
        l, t, r, b, f"{r-l}x{b-t}",
        bg_removed, status, str(p)
    ])
    crop_file.flush()

    count = get_processed_count(p.parent.name)
    status_label.config(
        text=f"💾 Saved as '{status}' — {p.parent.name}: {count}/{REQUIRED}",
        fg="green"
    )
    advance()


def on_save_already_correct():
    global final_img, pending_save_info
    if final_img is None or pending_save_info is None:
        return

    info       = pending_save_info
    p          = info["path"]
    w, h       = info["w"], info["h"]
    final_path = get_final_filename(p.parent.name)

    final_img.convert("RGBA").save(final_path, "PNG")

    crop_writer.writerow([
        p.name, p.parent.name, w, h, f"{w}x{h}",
        "", "", "", "", "already_correct",
        False, "already_correct", str(p)
    ])
    crop_file.flush()

    count = get_processed_count(p.parent.name)
    status_label.config(
        text=f"💾 Saved as 'already_correct' — {p.parent.name}: {count}/{REQUIRED}",
        fg="green"
    )
    advance()


def on_save_dispatch():
    if pending_save_info and pending_save_info.get("already_correct"):
        on_save_already_correct()
    else:
        on_save()


# -------------------------
# Redo
# -------------------------
def on_redo():
    global crop_coords, rect, preview_img, final_img
    global prev_final_img, original_cropped, pending_save_info, zoom_level

    crop_coords       = None
    preview_img       = None
    final_img         = None
    prev_final_img    = None
    original_cropped  = None
    pending_save_info = None
    zoom_level        = 1.0

    if rect:
        orig_canvas.delete(rect)
        rect = None

    preview_canvas.delete("all")
    preview_canvas.create_text(
        TARGET_SIZE // 2, TARGET_SIZE // 2,
        text="Preview will appear here",
        fill="gray", font=("Arial", 12)
    )

    coords_label.config(text="No crop set", fg="blue")
    confirm_btn.config(state="disabled")
    save_btn.config(state="disabled")
    redo_btn.config(state="disabled")
    status_label.config(text="")
    zoom_label.config(text="Zoom: 1.0x")
    reset_tools()


# -------------------------
# Skip / Invalid / Already Correct
# -------------------------
def on_skip():
    advance()


def on_invalid():
    info = orig_canvas.image_info
    p    = info["path"]
    w, h = info["w"], info["h"]

    replacement = get_replacement(p.parent.name)
    invalid_writer.writerow([
        p.name, p.parent.name, w, h, f"{w}x{h}", str(p), replacement
    ])
    invalid_file.flush()

    if replacement:
        repl_path = Path(replacement)
        if str(repl_path) not in processed:
            queue.insert(current_index + 1, repl_path)
        status_label.config(
            text=f"⚠️ Invalid. Next up: {repl_path.name}", fg="red"
        )
    else:
        status_label.config(text="⚠️ Invalid. No replacement found.", fg="red")
    advance()


def on_already_correct():
    info = orig_canvas.image_info
    p    = info["path"]
    w, h = info["w"], info["h"]

    save_info = {
        "path":            p,
        "w":               w,
        "h":               h,
        "crop":            (0, 0, w, h),
        "bg_removed":      False,
        "already_correct": True
    }
    show_final_preview(info["img"], save_info)
    status_label.config(
        text="Already Correct — confirm 256x256 result then Save",
        fg="blue"
    )


# -------------------------
# Build UI
# -------------------------
root = tk.Tk()
root.title("Crop + Process Tool")

screen_w = root.winfo_screenwidth()
screen_h = root.winfo_screenheight()
root.geometry(f"{min(1200, screen_w)}x{min(screen_h - 50, 850)}+0+0")

info_label = tk.Label(root, text="", font=("Arial", 10), pady=4)
info_label.pack(side="top", fill="x")

main_frame = tk.Frame(root)
main_frame.pack(side="top", fill="both", expand=True, padx=10)

# Left — original
left_frame = tk.LabelFrame(main_frame, text="Original", font=("Arial", 10))
left_frame.pack(side="left", fill="both", expand=True, padx=5, pady=5)

scroll_y_l = tk.Scrollbar(left_frame, orient="vertical")
scroll_y_l.pack(side="right", fill="y")
scroll_x_l = tk.Scrollbar(left_frame, orient="horizontal")
scroll_x_l.pack(side="bottom", fill="x")

orig_canvas = tk.Canvas(
    left_frame, cursor="cross", bg="gray",
    yscrollcommand=scroll_y_l.set,
    xscrollcommand=scroll_x_l.set
)
orig_canvas.pack(side="left", fill="both", expand=True)
scroll_y_l.config(command=orig_canvas.yview)
scroll_x_l.config(command=orig_canvas.xview)

orig_canvas.bind("<ButtonPress-1>",   on_press)
orig_canvas.bind("<B1-Motion>",       on_drag)
orig_canvas.bind("<ButtonRelease-1>", on_release)

# Right — preview + tools
right_frame = tk.LabelFrame(
    main_frame,
    text=f"Final Preview ({TARGET_SIZE}x{TARGET_SIZE})",
    font=("Arial", 10)
)
right_frame.pack(side="left", fill="both", expand=True, padx=5, pady=5)

# Preview canvas with scrollbars
preview_scroll_y = tk.Scrollbar(right_frame, orient="vertical")
preview_scroll_y.pack(side="right", fill="y")
preview_scroll_x = tk.Scrollbar(right_frame, orient="horizontal")
preview_scroll_x.pack(side="bottom", fill="x")

preview_canvas = tk.Canvas(
    right_frame, bg="white",
    width=TARGET_SIZE, height=TARGET_SIZE,
    yscrollcommand=preview_scroll_y.set,
    xscrollcommand=preview_scroll_x.set
)
preview_canvas.pack(pady=5)
preview_scroll_y.config(command=preview_canvas.yview)
preview_scroll_x.config(command=preview_canvas.xview)

preview_canvas.bind("<Button-1>",   on_preview_click)
preview_canvas.bind("<B1-Motion>",  on_preview_drag)
preview_canvas.bind("<MouseWheel>", on_preview_scroll)
preview_canvas.bind("<Button-4>",   on_preview_scroll)
preview_canvas.bind("<Button-5>",   on_preview_scroll)

# Zoom label
zoom_label = tk.Label(right_frame, text="Zoom: 1.0x", font=("Arial", 9), fg="gray")
zoom_label.pack()

# Touch-up tools row
tools_frame = tk.Frame(right_frame)
tools_frame.pack()

tk.Label(tools_frame, text="Touch-up:", font=("Arial", 10)).pack(side="left", padx=4)

wand_btn = tk.Button(
    tools_frame, text="🪄 Magic Wand",
    command=on_wand_toggle,
    state="disabled",
    bg="#9C27B0", fg="white",
    font=("Arial", 10, "bold"), width=13
)
wand_btn.pack(side="left", padx=3)

range_btn = tk.Button(
    tools_frame, text="🎨 Color Range",
    command=on_range_toggle,
    state="disabled",
    bg="#9C27B0", fg="white",
    font=("Arial", 10, "bold"), width=13
)
range_btn.pack(side="left", padx=3)

brush_btn = tk.Button(
    tools_frame, text="🖌️ Restore Brush",
    command=on_brush_toggle,
    state="disabled",
    bg="#795548", fg="white",
    font=("Arial", 10, "bold"), width=14
)
brush_btn.pack(side="left", padx=3)

erase_btn = tk.Button(
    tools_frame, text="🧹 Erase Brush",
    command=on_erase_toggle,
    state="disabled",
    bg="#F44336", fg="white",
    font=("Arial", 10, "bold"), width=13
)
erase_btn.pack(side="left", padx=3)

undo_btn = tk.Button(
    tools_frame, text="↩ Undo",
    command=on_undo,
    state="disabled",
    bg="gray", fg="white",
    font=("Arial", 10, "bold"), width=8
)
undo_btn.pack(side="left", padx=3)

# Sliders
sliders_frame = tk.Frame(right_frame)
sliders_frame.pack(pady=3)

tk.Label(sliders_frame, text="Tolerance:", font=("Arial", 10)).pack(side="left")
tolerance_var = tk.IntVar(value=TOLERANCE)
tk.Scale(
    sliders_frame, from_=5, to=100,
    orient="horizontal", variable=tolerance_var,
    length=140, font=("Arial", 9)
).pack(side="left", padx=5)

tk.Label(sliders_frame, text="Brush Size:", font=("Arial", 10)).pack(side="left")
brush_size_var = tk.IntVar(value=BRUSH_SIZE)
tk.Scale(
    sliders_frame, from_=1, to=40,
    orient="horizontal", variable=brush_size_var,
    length=140, font=("Arial", 9)
).pack(side="left", padx=5)

# Bottom bar
bottom = tk.Frame(root, pady=6)
bottom.pack(side="bottom", fill="x")

coords_label = tk.Label(bottom, text="No crop set", font=("Arial", 10), fg="blue")
coords_label.pack()

status_label = tk.Label(bottom, text="", font=("Arial", 10))
status_label.pack()

bg_remove_var = tk.BooleanVar(value=False)
tk.Checkbutton(
    bottom, text="☐ Remove Background",
    variable=bg_remove_var,
    font=("Arial", 11)
).pack(pady=3)

btn_frame = tk.Frame(bottom)
btn_frame.pack()

confirm_btn = tk.Button(
    btn_frame, text="👁 Preview",
    command=on_confirm,
    state="disabled",
    bg="#2196F3", fg="white",
    font=("Arial", 11, "bold"), width=12
)
confirm_btn.pack(side="left", padx=5)

save_btn = tk.Button(
    btn_frame, text="💾 Save",
    command=on_save_dispatch,
    state="disabled",
    bg="green", fg="white",
    font=("Arial", 11, "bold"), width=12
)
save_btn.pack(side="left", padx=5)

redo_btn = tk.Button(
    btn_frame, text="↩ Redo",
    command=on_redo,
    state="disabled",
    bg="purple", fg="white",
    font=("Arial", 11, "bold"), width=12
)
redo_btn.pack(side="left", padx=5)

skip_btn = tk.Button(
    btn_frame, text="⏭ Skip",
    command=on_skip,
    bg="orange", fg="white",
    font=("Arial", 11, "bold"), width=12
)
skip_btn.pack(side="left", padx=5)

invalid_btn = tk.Button(
    btn_frame, text="❌ Invalid",
    command=on_invalid,
    bg="red", fg="white",
    font=("Arial", 11, "bold"), width=12
)
invalid_btn.pack(side="left", padx=5)

already_btn = tk.Button(
    btn_frame, text="✔️ Already Correct",
    command=on_already_correct,
    bg="blue", fg="white",
    font=("Arial", 11, "bold"), width=16
)
already_btn.pack(side="left", padx=5)

load_image(current_index)
root.mainloop()
crop_file.close()
invalid_file.close()

Found:     7,550
Processed: 1,337
Remaining: 6,373



In [4]:
import numpy as np
from PIL import Image
from scipy import ndimage
import os

# ── Config ──────────────────────────────────────────────
input_base_dir  = r"G:\My Drive\Machine Learning Group Project\Processed Images"
output_base_dir = r"G:\My Drive\Machine Learning Group Project\Final Images"

# Set your Pokédex range here (use strings with leading zeros)
pokedex_start = "022"
pokedex_end   = "024"

# ── Clean Function ───────────────────────────────────────
def remove_stray_pixels(input_path, output_path):
    img = Image.open(input_path).convert("RGBA")
    arr = np.array(img)

    alpha = arr[:, :, 3]
    mask = alpha > 10

    labeled, num_features = ndimage.label(mask)

    if num_features == 0:
        print(f"  Skipped (no visible pixels): {input_path}")
        return

    region_sizes = ndimage.sum(mask, labeled, range(1, num_features + 1))
    largest_label = np.argmax(region_sizes) + 1

    clean_mask = labeled == largest_label
    arr[:, :, 3] = np.where(clean_mask, alpha, 0)

    Image.fromarray(arr).save(output_path)

# ── Main Loop ────────────────────────────────────────────
all_folders = sorted(os.listdir(input_base_dir))

for folder_name in all_folders:
    try:
        folder_num = folder_name.split(" - ")[0].strip()  # e.g. "004"
    except ValueError:
        continue

    # Compare as zero-padded strings
    if not (pokedex_start <= folder_num <= pokedex_end):
        continue

    input_folder  = os.path.join(input_base_dir, folder_name)
    output_folder = os.path.join(output_base_dir, folder_name)
    os.makedirs(output_folder, exist_ok=True)

    print(f"Processing: {folder_name}")

    for file in os.listdir(input_folder):
        if file.lower().endswith(".png"):
            input_path  = os.path.join(input_folder, file)
            output_path = os.path.join(output_folder, file)
            remove_stray_pixels(input_path, output_path)
            print(f"  Cleaned: {file}")

print("\nDone!")

Processing: 022 - Fearow
  Cleaned: Final 01.png
  Cleaned: Final 02.png
  Cleaned: Final 03.png
  Cleaned: Final 04.png
  Cleaned: Final 05.png
  Cleaned: Final 06.png
  Cleaned: Final 07.png
  Cleaned: Final 08.png
  Cleaned: Final 09.png
  Cleaned: Final 10.png
  Cleaned: Final 11.png
  Cleaned: Final 12.png
  Cleaned: Final 13.png
  Cleaned: Final 14.png
  Cleaned: Final 15.png
  Cleaned: Final 16.png
  Cleaned: Final 17.png
  Cleaned: Final 18.png
  Cleaned: Final 19.png
  Cleaned: Final 20.png
  Cleaned: Final 21.png
  Cleaned: Final 22.png
  Cleaned: Final 23.png
  Cleaned: Final 24.png
  Cleaned: Final 25.png
  Cleaned: Final 26.png
  Cleaned: Final 27.png
  Cleaned: Final 28.png
  Cleaned: Final 29.png
  Cleaned: Final 30.png
  Cleaned: Final 31.png
  Cleaned: Final 32.png
  Cleaned: Final 33.png
  Cleaned: Final 34.png
  Cleaned: Final 35.png
  Cleaned: Final 36.png
  Cleaned: Final 37.png
  Cleaned: Final 38.png
  Cleaned: Final 39.png
  Cleaned: Final 40.png
  Cleaned: Fina

In [6]:
import os
import shutil
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
from torchvision import transforms
from tqdm import tqdm

# ── Config ──────────────────────────────────────────────
data_dir   = r"G:\My Drive\Machine Learning Group Project\Final Images"
output_dir = r"G:\My Drive\Machine Learning Group Project"
npy_dir    = output_dir
img_size   = (256, 256)

# Set your Pokédex range here (use strings with leading zeros)
pokedex_start = "001"
pokedex_end   = "024"

# Number of augmented images per original (training only)
num_augments = 5

# ── Augmentation Pipeline ────────────────────────────────
augment = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandomResizedCrop(size=(256, 256), scale=(0.85, 1.0)),
    transforms.RandomGrayscale(p=0.2),
])

# ── Helper: Check if folder is already split ─────────────
def is_already_split(folder_path):
    return (
        os.path.isdir(os.path.join(folder_path, "Train")) and
        os.path.isdir(os.path.join(folder_path, "Val"))   and
        os.path.isdir(os.path.join(folder_path, "Test"))
    )

# ── Load Original Image Paths ────────────────────────────
image_paths = []
labels      = []
folder_map  = {}

all_folders   = sorted(os.listdir(data_dir))
valid_folders = [
    f for f in all_folders
    if os.path.isdir(os.path.join(data_dir, f))
    and pokedex_start <= f.split(" - ")[0].strip() <= pokedex_end
]

# Separate already-split folders from ones that still need processing
needs_split   = []
already_split = []

for folder_name in valid_folders:
    folder_path = os.path.join(data_dir, folder_name)
    if is_already_split(folder_path):
        already_split.append(folder_name)
        print(f"  ✓ Already split, skipping: {folder_name}")
    else:
        needs_split.append(folder_name)

# ── Load paths for folders that need splitting ───────────
for folder_name in tqdm(needs_split, desc="Loading folders"):
    folder_path = os.path.join(data_dir, folder_name)
    label = folder_name.split(" - ")[1].lower()

    for file in os.listdir(folder_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            full_path = os.path.join(folder_path, file)
            image_paths.append(full_path)
            labels.append(label)
            folder_map[full_path] = folder_name

# ── Split Paths (only for folders that need it) ──────────
if needs_split:
    print("\nSplitting data...")
    with tqdm(total=2, desc="Splitting") as pbar:
        paths_train, paths_temp, y_train_new, y_temp = train_test_split(
            image_paths, labels, test_size=0.40, stratify=labels, random_state=1996
        )
        pbar.update(1)
        paths_val, paths_test, y_val_new, y_test_new = train_test_split(
            paths_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=1996
        )
        pbar.update(1)

    # ── Move Val and Test (no augmentation) ─────────────
    def move_images(paths, split_name, desc):
        for path in tqdm(paths, desc=desc):
            folder_name = folder_map[path]
            dest_folder = os.path.join(data_dir, folder_name, split_name)
            os.makedirs(dest_folder, exist_ok=True)
            shutil.move(path, os.path.join(dest_folder, os.path.basename(path)))

    move_images(paths_val,  "Val",  "Moving val images ")
    move_images(paths_test, "Test", "Moving test images")

    # ── Move Train + Augment ─────────────────────────────
    print("\nMoving and augmenting training images...")
    for path in tqdm(paths_train, desc="Augmenting train"):
        folder_name = folder_map[path]
        dest_folder = os.path.join(data_dir, folder_name, "Train")
        os.makedirs(dest_folder, exist_ok=True)

        new_path = os.path.join(dest_folder, os.path.basename(path))
        shutil.move(path, new_path)

        img = Image.open(new_path).convert("RGB")
        base_name = os.path.splitext(os.path.basename(new_path))[0]
        for i in range(1, num_augments + 1):
            augmented = augment(img)
            augmented.save(os.path.join(dest_folder, f"{base_name}_aug{i}.png"))
else:
    print("\nAll folders already split — skipping to array loading!")

# ── Load ALL folders (split + already split) into Arrays ─
def load_images_from_split(split_name, desc):
    images = []
    labels = []
    for folder_name in tqdm(valid_folders, desc=desc):
        split_folder = os.path.join(data_dir, folder_name, split_name)
        label = folder_name.split(" - ")[1].lower()
        if not os.path.isdir(split_folder):
            continue
        for file in os.listdir(split_folder):
            if file.lower().endswith((".png", ".jpg", ".jpeg")):
                img = Image.open(os.path.join(split_folder, file)).convert("RGB")
                img = img.resize(img_size)
                images.append(np.array(img))
                labels.append(label)
    return (np.array(images) / 255.0).astype("float32"), np.array(labels)

print("\nLoading split images into arrays...")
X_train, y_train = load_images_from_split("Train", "Loading train")
X_val,   y_val   = load_images_from_split("Val",   "Loading val  ")
X_test,  y_test  = load_images_from_split("Test",  "Loading test ")

# ── Flatten ──────────────────────────────────────────────
print("\nFlattening...")
with tqdm(total=3, desc="Flattening") as pbar:
    X_train_flat = X_train.reshape(len(X_train), -1)
    pbar.update(1)
    X_val_flat = X_val.reshape(len(X_val), -1)
    pbar.update(1)
    X_test_flat = X_test.reshape(len(X_test), -1)
    pbar.update(1)

# ── Save to NPY ──────────────────────────────────────────
print("\nSaving .npy files...")
np.save(os.path.join(npy_dir, "X_train.npy"), X_train_flat)
print("  ✓ X_train.npy")
np.save(os.path.join(npy_dir, "X_val.npy"),   X_val_flat)
print("  ✓ X_val.npy")
np.save(os.path.join(npy_dir, "X_test.npy"),  X_test_flat)
print("  ✓ X_test.npy")
np.save(os.path.join(npy_dir, "y_train.npy"), y_train)
print("  ✓ y_train.npy")
np.save(os.path.join(npy_dir, "y_val.npy"),   y_val)
print("  ✓ y_val.npy")
np.save(os.path.join(npy_dir, "y_test.npy"),  y_test)
print("  ✓ y_test.npy")

# ── Verify ───────────────────────────────────────────────
print(f"\nTrain: {X_train.shape} | {y_train.shape}")
print(f"Val:   {X_val.shape}   | {y_val.shape}")
print(f"Test:  {X_test.shape}  | {y_test.shape}")
print("\nDone!")

  ✓ Already split, skipping: 001 - Bulbasaur
  ✓ Already split, skipping: 002 - Ivysaur
  ✓ Already split, skipping: 003 - Venusaur
  ✓ Already split, skipping: 004 - Charmander
  ✓ Already split, skipping: 005 - Charmeleon
  ✓ Already split, skipping: 006 - Charizard
  ✓ Already split, skipping: 007 - Squirtle
  ✓ Already split, skipping: 008 - Wartortle
  ✓ Already split, skipping: 009 - Blastoise
  ✓ Already split, skipping: 010 - Caterpie
  ✓ Already split, skipping: 011 - Metapod
  ✓ Already split, skipping: 012 - Butterfree
  ✓ Already split, skipping: 013 - Weedle
  ✓ Already split, skipping: 014 - Kakuna
  ✓ Already split, skipping: 015 - Beedrill
  ✓ Already split, skipping: 016 - Pidgey
  ✓ Already split, skipping: 017 - Pidgeotto
  ✓ Already split, skipping: 018 - Pidgeot
  ✓ Already split, skipping: 019 - Rattata
  ✓ Already split, skipping: 020 - Raticate
  ✓ Already split, skipping: 021 - Spearow
  ✓ Already split, skipping: 022 - Fearow
  ✓ Already split, skipping: 023 -

Loading folders: 0it [00:00, ?it/s]



All folders already split — skipping to array loading!

Loading split images into arrays...


Loading test : 100%|██████████| 24/24 [00:06<00:00,  3.43it/s]



Flattening...


Flattening: 100%|██████████| 3/3 [00:00<00:00, 35.08it/s]


Saving .npy files...


  ✓ X_train.npy
  ✓ X_val.npy
  ✓ X_test.npy
  ✓ y_train.npy
  ✓ y_val.npy
  ✓ y_test.npy

Train: (4320, 256, 256, 3) | (4320,)
Val:   (240, 256, 256, 3)   | (240,)
Test:  (240, 256, 256, 3)  | (240,)

Done!


Removing augmented files...
Done! Removed 5250 augmented files.
